# 04 — SQL Analytics: WO-1 Readiness Audit

## Purpose

WO-1 establishes the verified baseline for analytical modelling. It reconciles the existing quality registry and STAGING row counts, inspects current database structures, validates student-presentation keys, and measures assessment-join fan-out.

Canonical SQL definitions live in `sql/04_analytics/`. This notebook invokes them by query ID through `src/analytics/db.py`; SQL is not duplicated here.

## Execution controls

- Each query uses a separate PostgreSQL connection with read-only session settings, a 120-second statement timeout, and a 5-second lock timeout.
- Displayed results are capped at 500 rows. Exact counts are computed in PostgreSQL; the full `student_vle` table is not loaded into Pandas.
- RAW loading, STAGING creation, quality-registry seeding, and DDL are outside this work order.
- AN2.01 and AN2.02 depend on successful key validation in AN1.01.
- Historical counts are reference values, not evidence of the current database state.

## Work-order status

**Status:** Prepared for local execution. No live WO-1 findings have been validated yet.

## 0. Query helper and connection controls

The helper resolves canonical SQL files by stable query ID, opens a separate read-only database session for each query, and returns a bounded result table with execution time. Connection settings are read from the local environment and are never displayed.

In [2]:
# 1. Load the project root and canonical query runner.
from pathlib import Path
import sys
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "sql" / "04_analytics").exists():
    PROJECT_ROOT = Path.cwd().parent
if not (PROJECT_ROOT / "sql" / "04_analytics").exists():
    raise FileNotFoundError("Open this notebook from the repository or notebooks folder.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics.db import run_query

query_results = {}
print("Project root:", PROJECT_ROOT.name)
print("Canonical SQL folder found:", (PROJECT_ROOT / "sql" / "04_analytics").exists())

Project root: OULAD-EdTech-Learning-Analytics
Canonical SQL folder found: True


In [3]:
print("run_query defined:", "run_query" in globals())
print("query_results defined:", "query_results" in globals())
print("Existing results in memory:", len(query_results))

run_query defined: True
query_results defined: True
Existing results in memory: 0


**Validation status:** The import and path checks establish whether the canonical SQL directory and query helper are available. No database result is produced by this cell.

## AN0.01 — Database identity and session safety

**Question:** Is the active connection using the intended database and read-only session settings?

**Expectation:** The database is `oulad_analytics`; read-only mode is enabled.

**Interpretation:** A database or session-setting mismatch invalidates subsequent results and must be resolved before the audit proceeds.

In [2]:
# 1. Verify the active database and read-only session before continuing.
result = run_query("AN0.01")
query_results["AN0.01"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

row = result["data"].iloc[0]
if row["database_name"] != "oulad_analytics" or str(row["transaction_read_only"]).lower() != "on":
    raise RuntimeError("Safety gate failed. Stop the notebook and investigate the active connection.")

,database_name,database_user,current_schema,server_version,transaction_read_only,default_transaction_read_only,statement_timeout,lock_timeout
0,oulad_analytics,postgres,public,18.6,on,on,2min,5s


Query file: sql\04_analytics\AN0.01_database_identity.sql
Elapsed seconds: 0.026
Displayed rows: 1; truncated: False


### Interpretation and decision — AN0.01

**Observed result:** Pending local execution.

**Decision rule:** Continue only when the database identity is correct and the session confirms read-only operation. Otherwise, suspend WO-1 and investigate the connection configuration.

## AN0.02 — Existing quality registry snapshot

**Question:** Does the live registry match the documented historical record?

**Historical reference:** 29 records and 29 distinct test codes; 25 `PASS`, 4 `REVIEW`, and 0 `FAIL`. Historical `executed_at` values may be NULL.

**Interpretation:** A difference is a reconciliation issue. This query reads the registry; it does not rerun or reseed completed engineering checks.

In [3]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN0.02")
query_results["AN0.02"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,metric_name,metric_value
0,distinct_test_codes,29
1,duplicate_test_code_excess,0
2,fail_rows,0
3,null_executed_at_rows,29
4,pass_rows,25
5,registry_rows,29
6,review_rows,4


Query file: sql\04_analytics\AN0.02_quality_registry_snapshot.sql
Elapsed seconds: 0.098
Displayed rows: 7; truncated: False


### Interpretation and decision — AN0.02

**Observed result:** Pending local execution.

**Decision rule:** Reconcile any differences against existing registry contents and documented execution history. Do not reconstruct missing historical timestamps or rerun seed operations as a substitute for reconciliation.

## AN0.03 — STAGING row-count reconciliation

**Question:** Do current STAGING row counts match the established baseline?

**Historical reference:** `courses` 22; `assessments` 206; `student_info` 32,593; `student_registration` 32,593; `student_assessment` 173,912; `vle` 6,364; `student_vle` 10,655,280.

**Interpretation:** PostgreSQL computes exact counts and returns seven summary rows. The large `student_vle` scan is subject to the configured timeout; no table data is transferred to Pandas.

In [4]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN0.03")
query_results["AN0.03"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,table_name,expected_rows,actual_rows,row_count_difference,reconciliation_status
0,assessments,206,206,0,MATCH
1,courses,22,22,0,MATCH
2,student_assessment,173912,173912,0,MATCH
3,student_info,32593,32593,0,MATCH
4,student_registration,32593,32593,0,MATCH
5,student_vle,10655280,10655280,0,MATCH
6,vle,6364,6364,0,MATCH


Query file: sql\04_analytics\AN0.03_staging_row_count_reconciliation.sql
Elapsed seconds: 3.651
Displayed rows: 7; truncated: False


### Interpretation and decision — AN0.03

**Observed result:** Pending local execution.

**Decision rule:** Matching counts support continuity with the historical baseline but do not establish semantic correctness. Any discrepancy requires a targeted read-only investigation before dependent analysis proceeds.

## AN0.04 — STAGING sizes and indexes

**Question:** What physical relation sizes and indexes exist in the current database?

**Expectation:** The live catalog, not assumptions about CTAS, determines which indexes exist.

**Interpretation:** Record relation sizes and existing index definitions as a performance baseline. This query creates no indexes or other database objects.

In [5]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN0.04")
query_results["AN0.04"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,schema_name,table_name,table_size,total_size_including_indexes,index_count,index_names
0,staging,student_vle,612 MB,612 MB,0,(none)
1,staging,student_assessment,9216 kB,9256 kB,0,(none)
2,staging,student_info,3720 kB,3760 kB,0,(none)
3,staging,student_registration,1664 kB,1704 kB,0,(none)
4,staging,vle,336 kB,376 kB,0,(none)
5,staging,assessments,16 kB,24 kB,0,(none)
6,staging,courses,8192 bytes,16 kB,0,(none)


Query file: sql\04_analytics\AN0.04_staging_relation_sizes_and_indexes.sql
Elapsed seconds: 0.304
Displayed rows: 7; truncated: False


### Interpretation and decision — AN0.04

**Observed result:** Pending local execution.

**Decision rule:** Use measured sizes and existing index definitions to assess the cost of later analytical queries. Any proposal to add an index requires a separate cost-benefit review and approval.

## AN1.01 — Student key completeness and uniqueness

**Question:** Is `(code_module, code_presentation, id_student)` complete and unique in both `student_info` and `student_registration`?

**Expectation:** No key components are NULL and no duplicate key groups exist.

**Interpretation:** AN2 profile queries are valid only if both tables pass this key check.

In [6]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN1.01")
query_results["AN1.01"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,table_name,total_rows,null_key_rows,distinct_key_values,duplicate_key_groups,excess_rows,gate_status
0,student_info,32593,0,32593,0,0,PASS
1,student_registration,32593,0,32593,0,0,PASS


Query file: sql\04_analytics\AN1.01_student_key_uniqueness.sql
Elapsed seconds: 0.189
Displayed rows: 2; truncated: False


### Interpretation and decision — AN1.01

**Observed result:** Pending local execution.

**Decision rule:** Proceed to AN1.02 and the gated AN2 queries only if both tables satisfy key completeness and uniqueness. Otherwise, investigate the affected key groups before using student-presentation grain.

## AN1.02 — Student key-set reconciliation

**Question:** Do `student_info` and `student_registration` contain the same composite-key population, and how do their distinct student counts compare?

**Expectation:** No keys are exclusive to either table if both represent the same student-presentation population.

**Interpretation:** Read this result alongside AN1.01; equal row counts alone do not prove that the key sets match.

In [7]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN1.02")
query_results["AN1.02"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,metric_name,metric_value
0,shared_complete_keys,32593
1,student_info_distinct_student_ids,28785
2,student_info_only_complete_keys,0
3,student_registration_distinct_student_ids,28785
4,student_registration_only_complete_keys,0


Query file: sql\04_analytics\AN1.02_student_key_set_reconciliation.sql
Elapsed seconds: 0.252
Displayed rows: 5; truncated: False


### Interpretation and decision — AN1.02

**Observed result:** Pending local execution.

**Decision rule:** Any key-set difference requires identifying which table contains the unmatched keys and whether the difference reflects population definitions, missing records, or a data issue. Do not infer equivalence from equal totals alone.

## AN1.03 — Assessment join fan-out

**Question:** How much does joining `student_assessment` to `student_info` by `id_student` alone inflate the row count compared with the assessment-metadata path?

**Expectation:** The student-only join may multiply rows because one student can appear in multiple module presentations. The defensible route joins through `assessments` and matches the module and presentation.

**Interpretation:** A multiplier above 1 indicates join fan-out, not additional assessment records. Unmatched records must also be considered.

In [8]:
# 1. Execute the canonical SQL file and display only its bounded result set.
result = run_query("AN1.03")
query_results["AN1.03"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,source_rows,naive_join_rows,correct_join_rows,naive_join_multiplier,correct_join_multiplier,unmatched_assessment_rows,unmatched_student_presentation_rows,matched_student_presentation_rows
0,173912,207319,173912,1.1921,1.0000,0,0,173912


Query file: sql\04_analytics\AN1.03_student_assessment_join_fanout.sql
Elapsed seconds: 0.296
Displayed rows: 1; truncated: False


### Interpretation and decision — AN1.03

**Observed result:** Pending local execution.

**Decision rule:** Use the assessment-metadata path for student-presentation analysis. Record the measured fan-out and unmatched counts; do not use the student-only join for outcome metrics.

## AN2.01 — Student counts by presentation

**Question:** Do `student_info` and `student_registration` counts reconcile at module-presentation grain?

**Expectation:** Counts should reconcile when both tables represent the same validated key population.

**Execution gate:** This query runs only when AN1.01 passes for both tables.

In [9]:
# 1. Run this profile only when both student tables passed AN1.01.
gate = query_results.get("AN1.01")
if gate is None or len(gate["data"]) != 2 or not (gate["data"]["gate_status"] == "PASS").all():
    print("SKIPPED: AN1.01 did not confirm complete, unique keys in both student tables.")
else:
    result = run_query("AN2.01")
    query_results["AN2.01"] = result
    display(result["data"])
    print(f"Query file: {result['query_file']}")
    print(f"Elapsed seconds: {result['elapsed_seconds']}")
    print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,student_info_rows,registration_rows,student_info_distinct_students,registration_distinct_students,row_count_difference
0,AAA,2013J,383,383,383,383,0
1,AAA,2014J,365,365,365,365,0
2,BBB,2013B,1767,1767,1767,1767,0
3,BBB,2013J,2237,2237,2237,2237,0
4,BBB,2014B,1613,1613,1613,1613,0
5,BBB,2014J,2292,2292,2292,2292,0
6,CCC,2014B,1936,1936,1936,1936,0
7,CCC,2014J,2498,2498,2498,2498,0
8,DDD,2013B,1303,1303,1303,1303,0
9,DDD,2013J,1938,1938,1938,1938,0


Query file: sql\04_analytics\AN2.01_student_presentation_profile.sql
Elapsed seconds: 0.036
Displayed rows: 22; truncated: False


### Interpretation and decision — AN2.01

**Observed result:** Pending local execution.

**Decision rule:** Interpret counts at module-presentation grain. Differences must be reconciled before selecting a population denominator for subsequent metrics.

## AN2.02 — Final-result distribution

**Question:** What final-result categories are observed within each module presentation, including missing outcomes?

**Expectation:** Within-presentation percentages sum to approximately 100%.

**Denominator:** All rows in `student_info` within the relevant module presentation, including missing `final_result` values.

**Interpretation:** This is a descriptive distribution; it does not establish causes of success, failure, or withdrawal.

In [10]:
# 1. Run this profile only when both student tables passed AN1.01.
gate = query_results.get("AN1.01")
if gate is None or len(gate["data"]) != 2 or not (gate["data"]["gate_status"] == "PASS").all():
    print("SKIPPED: AN1.01 did not confirm complete, unique keys in both student tables.")
else:
    result = run_query("AN2.02")
    query_results["AN2.02"] = result
    display(result["data"])
    print(f"Query file: {result['query_file']}")
    print(f"Elapsed seconds: {result['elapsed_seconds']}")
    print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,final_result_category,student_rows,presentation_student_rows,percentage_within_presentation
0,AAA,2013J,Distinction,20,383,5.22
1,AAA,2013J,Fail,45,383,11.75
2,AAA,2013J,Pass,258,383,67.36
3,AAA,2013J,Withdrawn,60,383,15.67
4,AAA,2014J,Distinction,24,365,6.58
...,...,...,...,...,...,...
83,GGG,2014B,Withdrawn,100,833,12.00
84,GGG,2014J,Distinction,127,749,16.96
85,GGG,2014J,Fail,179,749,23.90
86,GGG,2014J,Pass,317,749,42.32


Query file: sql\04_analytics\AN2.02_final_result_distribution.sql
Elapsed seconds: 0.055
Displayed rows: 88; truncated: False


### Interpretation and decision — AN2.02

**Observed result:** Pending local execution.

**Decision rule:** Verify denominators and percentages within each presentation. Missing outcomes remain visible and must not be silently excluded. The distribution describes observed outcomes and does not by itself explain why they occurred.

## AN1.04 — Course-presentation relationship validation

**Question:** Do student-presentation rows map to exactly one course metadata row?

**Expectation:** No duplicate course keys, no unmatched student-presentation rows, and a join multiplier of 1.0000.

**Grain:** One row per module-presentation relationship check.

**Source:** student_info and courses

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Use this relationship only if the measured key and join checks pass.

In [4]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN1.04")
query_results["AN1.04"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,diagnostic,table_name,source_rows,distinct_course_keys,matched_join_rows,unmatched_rows,duplicate_key_groups,duplicate_excess_rows,joined_rows,join_multiplier
0,COURSE_KEY,courses,22,22.0,NaN,NaN,0.0,0.0,NaN,None
1,STUDENT_COURSE_JOIN,student_info,32593,NaN,32593.0,0.0,NaN,NaN,32593.0,1.0000
2,STUDENT_COURSE_JOIN,student_registration,32593,NaN,32593.0,0.0,NaN,NaN,32593.0,1.0000


Query file: sql\04_analytics\AN1.04_course_presentation_relationships.sql
Elapsed seconds: 0.315
Displayed rows: 3; truncated: False


### Interpretation and decision — AN1.04

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** Review the observed counts, denominators, missing values, and row grain before drawing conclusions. Stop downstream course-level interpretation if duplicate keys, unmatched rows, or a multiplier other than 1 is observed.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## AN1.05 — Assessment metadata relationship validation

**Question:** Does each student_assessment row map to one assessment metadata row through the assessment identifier?

**Expectation:** No duplicate assessment identifiers, no unmatched submissions, and a join multiplier of 1.0000.

**Grain:** Assessment relationship and submission rows.

**Source:** student_assessment and assessments

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Use the metadata join path only after the reported cardinality checks pass.

In [5]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN1.05")
query_results["AN1.05"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,diagnostic,source_rows,distinct_assessment_ids,matched_join_rows,unmatched_rows,duplicate_key_groups,duplicate_excess_rows,joined_rows,join_multiplier
0,ASSESSMENT_KEY,206,206.0,NaN,NaN,0.0,0.0,NaN,None
1,SUBMISSION_JOIN,173912,NaN,173912.0,0.0,NaN,NaN,173912.0,1.0000


Query file: sql\04_analytics\AN1.05_assessment_metadata_relationships.sql
Elapsed seconds: 0.334
Displayed rows: 2; truncated: False


### Interpretation and decision — AN1.05

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** Review the observed counts, denominators, missing values, and row grain before drawing conclusions. Investigate duplicate or missing assessment metadata before interpreting submission metrics.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## AN1.06 — Registration grain versus distinct people

**Question:** How do student-presentation registration counts differ from distinct people?

**Expectation:** Registration rows should be greater than or equal to distinct people; presentation participation per person should be explicit.

**Grain:** Registration rows and distinct student IDs at the reported grouping.

**Source:** student_registration

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Do not treat registration records as unique people across presentations.

In [6]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN1.06")
query_results["AN1.06"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,metric_group,presentation_count,registration_rows,people_count,null_registration_date_rows,rows_with_unregistration_date
0,OVERALL,NaN,32593.0,28785,45.0,10072.0
1,PEOPLE_BY_PRESENTATION_COUNT,1.0,NaN,25247,NaN,NaN
2,PEOPLE_BY_PRESENTATION_COUNT,2.0,NaN,3293,NaN,NaN
3,PEOPLE_BY_PRESENTATION_COUNT,3.0,NaN,221,NaN,NaN
4,PEOPLE_BY_PRESENTATION_COUNT,4.0,NaN,23,NaN,NaN
5,PEOPLE_BY_PRESENTATION_COUNT,5.0,NaN,1,NaN,NaN


Query file: sql\04_analytics\AN1.06_registration_person_grain.sql
Elapsed seconds: 0.293
Displayed rows: 6; truncated: False


### Interpretation and decision — AN1.06

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** Review the observed counts, denominators, missing values, and row grain before drawing conclusions. Use registrations for presentation-level denominators and distinct student IDs only for person-level counts.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## AN1.07 — Activity coverage by presentation

**Question:** How much VLE activity is recorded in each registered presentation?

**Expectation:** One row per registered presentation, including registered students, active students, active days, sites, negative-day rows, and stored click sum.

**Grain:** Module-presentation aggregate.

**Source:** student_registration and student_vle

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Allow up to 600,000 ms for this activity-scale query; execute separately and review the returned timing.

**Read as:** Use the registered population as the denominator and preserve the recorded activity grain.

In [7]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN1.07", statement_timeout_ms=600000)
query_results["AN1.07"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,registration_rows,registered_students,activity_rows,students_with_any_activity,matched_active_students,activity_students_not_registered,pct_registered_students_with_matched_activity,distinct_sites_with_activity,distinct_activity_days,negative_day_rows,minimum_activity_day,maximum_activity_day,stored_sum_click
0,AAA,2013J,383,383,180982,378,378,0,98.69,209,279,10847,-10,268,648494
1,AAA,2014J,365,365,169316,357,357,0,97.81,197,294,14508,-24,269,598158
2,BBB,2013B,1767,1767,403266,1537,1537,0,86.98,315,250,19738,-9,240,1347911
3,BBB,2013J,2237,2237,452638,1870,1870,0,83.59,320,280,19826,-23,268,1378656
4,BBB,2014B,1613,1613,273236,1294,1294,0,80.22,311,242,11900,-9,234,833865
5,BBB,2014J,2292,2292,438424,1921,1921,0,83.81,206,272,27058,-9,262,1673744
6,CCC,2014B,1936,1936,496181,1681,1681,0,86.83,180,258,26648,-18,241,1889177
7,CCC,2014J,2498,2498,711646,2302,2302,0,92.15,220,288,42604,-18,269,2792972
8,DDD,2013B,1303,1303,536837,1214,1214,0,93.17,408,257,44913,-16,240,1387467
9,DDD,2013J,1938,1938,680806,1768,1768,0,91.23,456,280,59515,-18,261,1757214


Query file: sql\04_analytics\AN1.07_activity_coverage_by_presentation.sql
Elapsed seconds: 47.94
Displayed rows: 22; truncated: False


### Interpretation and decision — AN1.07

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** Review the observed counts, denominators, missing values, and row grain before drawing conclusions. Review negative dates and click totals as observed data characteristics; do not infer errors or deduplicate activity rows from this profile alone.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## AN1.08 — Registrations without activity

**Question:** Which registered students have no matching VLE activity rows in their presentation?

**Expectation:** Counts and rates should be reported at module-presentation grain using registrations as the denominator.

**Grain:** Registered student-presentation keys and matched activity keys.

**Source:** student_registration and student_vle

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Allow up to 600,000 ms for this activity-scale query; execute separately and review the returned timing.

**Read as:** Interpret rates only with the registration denominator and exact composite key.

In [8]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN1.08", statement_timeout_ms=600000)
query_results["AN1.08"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,registration_rows,registrations_with_activity,registrations_without_activity,pct_registrations_without_activity
0,AAA,2013J,383,378,5,1.31
1,AAA,2014J,365,357,8,2.19
2,BBB,2013B,1767,1537,230,13.02
3,BBB,2013J,2237,1870,367,16.41
4,BBB,2014B,1613,1294,319,19.78
5,BBB,2014J,2292,1921,371,16.19
6,CCC,2014B,1936,1681,255,13.17
7,CCC,2014J,2498,2302,196,7.85
8,DDD,2013B,1303,1214,89,6.83
9,DDD,2013J,1938,1768,170,8.77


Query file: sql\04_analytics\AN1.08_registrations_without_activity.sql
Elapsed seconds: 2.961
Displayed rows: 22; truncated: False


### Interpretation and decision — AN1.08

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** Review the observed counts, denominators, missing values, and row grain before drawing conclusions. Treat absence of activity rows as an observed coverage condition, not proof of non-participation in all learning activities.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## AN1.09 — VLE resource match coverage

**Question:** How many recorded activity rows match VLE resource metadata?

**Expectation:** Matched and unmatched activity rows should reconcile to the activity-row denominator.

**Grain:** Activity rows and resource metadata matches.

**Source:** student_vle and vle

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Allow up to 600,000 ms for this activity-scale query; execute separately and review the returned timing.

**Read as:** Report row-level match coverage and avoid treating unmatched records as zero activity.

In [9]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN1.09", statement_timeout_ms=600000)
query_results["AN1.09"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,diagnostic,match_status,row_count,distinct_site_ids,duplicate_groups,duplicate_excess_rows
0,ACTIVITY_RESOURCE_MATCH,EXACT_PRESENTATION_MATCH,10655280,NaN,NaN,NaN
1,VLE_KEY,NaN,6364,6364.0,0.0,0.0


Query file: sql\04_analytics\AN1.09_vle_resource_match_coverage.sql
Elapsed seconds: 17.263
Displayed rows: 2; truncated: False


### Interpretation and decision — AN1.09

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** Review the observed counts, denominators, missing values, and row grain before drawing conclusions. Investigate unmatched resource identifiers before assigning resource-level meaning; do not alter staging data.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## AN2.03 — Presentation size profile

**Question:** How does the student population vary across module presentations?

**Expectation:** One row per module-presentation with counts and rates derived from the validated course-presentation relationship.

**Grain:** Module-presentation aggregate.

**Source:** student_info and courses

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Use counts for scale comparisons; do not interpret size as a causal factor.

In [10]:
# 1. Run this profile only after AN1.04 confirms unique course keys and complete, one-to-one student-course matches.
gate = query_results.get("AN1.04")
if gate is None or gate["data"].empty:
    print("SKIPPED: AN1.04 has not returned course-presentation diagnostics.")
else:
    data = gate["data"]
    course_key = data.loc[data["diagnostic"] == "COURSE_KEY"]
    joins = data.loc[data["diagnostic"] == "STUDENT_COURSE_JOIN"]
    gate_passes = (
        len(course_key) == 1
        and len(joins) == 2
        and int(course_key.iloc[0]["duplicate_key_groups"]) == 0
        and (joins["unmatched_rows"] == 0).all()
        and (joins["join_multiplier"] == 1.0).all()
    )
    if not gate_passes:
        print("SKIPPED: AN1.04 did not confirm unique course keys and complete one-to-one joins.")
        display(data)
    else:
        result = run_query("AN2.03")
        query_results["AN2.03"] = result
        display(result["data"])
        print(f"Query file: {result['query_file']}")
        print(f"Elapsed seconds: {result['elapsed_seconds']}")
        print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,module_presentation_length,student_presentation_rows,distinct_students_in_presentation,within_presentation_key_status
0,AAA,2013J,268,383,383,MATCH
1,AAA,2014J,269,365,365,MATCH
2,BBB,2013B,240,1767,1767,MATCH
3,BBB,2013J,268,2237,2237,MATCH
4,BBB,2014B,234,1613,1613,MATCH
5,BBB,2014J,262,2292,2292,MATCH
6,CCC,2014B,241,1936,1936,MATCH
7,CCC,2014J,269,2498,2498,MATCH
8,DDD,2013B,240,1303,1303,MATCH
9,DDD,2013J,261,1938,1938,MATCH


Query file: sql\04_analytics\AN2.03_presentation_size_profile.sql
Elapsed seconds: 0.06
Displayed rows: 22; truncated: False


### Interpretation and decision — AN2.03

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** If the relationship gate does not pass, do not interpret this profile; resolve the relationship issue first.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## AN2.04 — Gender, age band, and disability distribution

**Question:** How are gender, age band, and disability status distributed across presentations?

**Expectation:** Category counts and percentages should reconcile to each presentation denominator, with missing categories visible.

**Grain:** Student-presentation records grouped by demographic categories.

**Source:** student_info

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** These are descriptive distributions and do not establish causal relationships.

In [11]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN2.04")
query_results["AN2.04"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,dimension_name,category_value,category_rows,presentation_rows,pct_within_presentation
0,AAA,2013J,age_band,0-35,179,383,46.74
1,AAA,2013J,age_band,35-55,178,383,46.48
2,AAA,2013J,age_band,55<=,26,383,6.79
3,AAA,2014J,age_band,0-35,173,365,47.40
4,AAA,2014J,age_band,35-55,173,365,47.40
...,...,...,...,...,...,...,...
148,GGG,2013J,gender,M,178,952,18.70
149,GGG,2014B,gender,F,666,833,79.95
150,GGG,2014B,gender,M,167,833,20.05
151,GGG,2014J,gender,F,603,749,80.51


Query file: sql\04_analytics\AN2.04_gender_age_disability_distribution.sql
Elapsed seconds: 0.133
Displayed rows: 153; truncated: False


### Interpretation and decision — AN2.04

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** Review the observed counts, denominators, missing values, and row grain before drawing conclusions. Keep missing values visible and state whether percentages use all students or non-missing values.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## AN2.05 — Region and prior-education distribution

**Question:** How do region and prior-education categories vary across presentations?

**Expectation:** Category counts and percentages should reconcile to the stated denominator, with missing values retained explicitly.

**Grain:** Student-presentation records grouped by region and prior education.

**Source:** student_info

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Describe composition only; do not infer regional or educational causes of outcomes.

In [12]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN2.05")
query_results["AN2.05"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,dimension_name,category_value,category_rows,presentation_rows,pct_within_presentation
0,AAA,2013J,highest_education,A Level or Equivalent,194,383,50.65
1,AAA,2013J,highest_education,HE Qualification,98,383,25.59
2,AAA,2013J,highest_education,Lower Than A Level,83,383,21.67
3,AAA,2013J,highest_education,Post Graduate Qualification,8,383,2.09
4,AAA,2014J,highest_education,A Level or Equivalent,201,365,55.07
...,...,...,...,...,...,...,...
383,GGG,2014J,region,South Region,72,749,9.61
384,GGG,2014J,region,South West Region,62,749,8.28
385,GGG,2014J,region,Wales,51,749,6.81
386,GGG,2014J,region,West Midlands Region,51,749,6.81


Query file: sql\04_analytics\AN2.05_region_education_distribution.sql
Elapsed seconds: 0.101
Displayed rows: 388; truncated: False


### Interpretation and decision — AN2.05

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** Review the observed counts, denominators, missing values, and row grain before drawing conclusions. Verify category and missing-value denominators before comparing percentages.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## AN2.06 — Numeric student profile

**Question:** What do the available numeric student-profile fields look like by presentation?

**Expectation:** Report valid counts and distribution summaries with missingness separated from observed numeric values.

**Grain:** Student-presentation records grouped by presentation.

**Source:** student_info

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Use summaries as descriptive context, not as validated predictive features.

In [13]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN2.06")
query_results["AN2.06"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,metric_name,total_rows,non_null_rows,null_rows,minimum_value,mean_value,median_value,maximum_value
0,AAA,2013J,num_of_prev_attempts,383,383,0,0,0.00,0.0,0
1,AAA,2014J,num_of_prev_attempts,365,365,0,0,0.11,0.0,1
2,BBB,2013B,num_of_prev_attempts,1767,1767,0,0,0.26,0.0,5
3,BBB,2013J,num_of_prev_attempts,2237,2237,0,0,0.21,0.0,5
4,BBB,2014B,num_of_prev_attempts,1613,1613,0,0,0.24,0.0,6
5,BBB,2014J,num_of_prev_attempts,2292,2292,0,0,0.16,0.0,5
6,CCC,2014B,num_of_prev_attempts,1936,1936,0,0,0.00,0.0,0
7,CCC,2014J,num_of_prev_attempts,2498,2498,0,0,0.09,0.0,1
8,DDD,2013B,num_of_prev_attempts,1303,1303,0,0,0.28,0.0,6
9,DDD,2013J,num_of_prev_attempts,1938,1938,0,0,0.20,0.0,5


Query file: sql\04_analytics\AN2.06_numeric_student_profile.sql
Elapsed seconds: 0.189
Displayed rows: 44; truncated: False


### Interpretation and decision — AN2.06

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** Review the observed counts, denominators, missing values, and row grain before drawing conclusions. Check ranges, valid-value counts, and denominators; do not silently impute missing values.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## AN2.07 — IMD missingness by presentation

**Question:** How does missing IMD band information vary by presentation?

**Expectation:** Missing and non-missing counts should sum to the presentation population, and the missing rate should use that population as denominator.

**Grain:** Student-presentation records grouped by presentation.

**Source:** student_info

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** A missingness rate describes data availability, not deprivation level.

In [14]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN2.07")
query_results["AN2.07"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,code_module,code_presentation,presentation_rows,imd_missing_rows,imd_present_rows,pct_imd_missing
0,FFF,2013B,1614,95,1519,5.89
1,CCC,2014J,2498,146,2352,5.84
2,CCC,2014B,1936,108,1828,5.58
3,EEE,2013J,1052,52,1000,4.94
4,FFF,2014B,1500,72,1428,4.80
5,DDD,2014B,1228,58,1170,4.72
6,DDD,2013B,1303,61,1242,4.68
7,DDD,2014J,1803,81,1722,4.49
8,FFF,2013J,2283,101,2182,4.42
9,FFF,2014J,2365,99,2266,4.19


Query file: sql\04_analytics\AN2.07_imd_missing_by_presentation.sql
Elapsed seconds: 0.034
Displayed rows: 22; truncated: False


### Interpretation and decision — AN2.07

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** Review the observed counts, denominators, missing values, and row grain before drawing conclusions. Keep missingness explicit and verify counts against the full presentation population.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## AN2.08 — IMD missingness by region

**Question:** Does IMD band missingness vary across region categories?

**Expectation:** Report missing and non-missing counts with the denominator for each region category clearly defined.

**Grain:** Student-presentation records grouped by region.

**Source:** student_info

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Do not interpret missing IMD values as low or high deprivation.

In [15]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN2.08")
query_results["AN2.08"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,region_category,student_presentation_rows,imd_missing_rows,imd_present_rows,pct_imd_missing
0,North Region,1823,731,1092,40.10
1,Ireland,1184,266,918,22.47
2,South Region,3092,48,3044,1.55
3,West Midlands Region,2582,39,2543,1.51
4,Scotland,3446,12,3434,0.35
5,Yorkshire Region,2006,5,2001,0.25
6,South West Region,2436,5,2431,0.21
7,North Western Region,2906,5,2901,0.17
8,East Anglian Region,3340,0,3340,0.00
9,London Region,3216,0,3216,0.00


Query file: sql\04_analytics\AN2.08_imd_missing_by_region.sql
Elapsed seconds: 0.023
Displayed rows: 13; truncated: False


### Interpretation and decision — AN2.08

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** Review the observed counts, denominators, missing values, and row grain before drawing conclusions. Inspect missing region values and small denominators before comparing rates.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## AN2.09 — IMD missingness by final result

**Question:** Does IMD band missingness differ across final-result categories?

**Expectation:** Missing and non-missing counts and rates should be reported for each outcome category, including missing outcomes if represented.

**Grain:** Student-presentation records grouped by final result.

**Source:** student_info

**Safety:** READ-ONLY. The canonical SQL is stored in `sql/04_analytics/`; this notebook calls it by query ID. No RAW or STAGING data is modified.

**Runtime:** Use the query runner's configured read-only timeout.

**Read as:** Do not infer that IMD missingness causes a final result.

In [16]:
# 1. Execute the canonical read-only SQL file and retain its bounded output.
result = run_query("AN2.09")
query_results["AN2.09"] = result
display(result["data"])
print(f"Query file: {result['query_file']}")
print(f"Elapsed seconds: {result['elapsed_seconds']}")
print(f"Displayed rows: {result['rows_displayed']}; truncated: {result['display_truncated']}")

,final_result_category,student_presentation_rows,imd_missing_rows,imd_present_rows,pct_imd_missing
0,Distinction,3024,199,2825,6.58
1,Pass,12361,531,11830,4.30
2,Withdrawn,10156,236,9920,2.32
3,Fail,7052,145,6907,2.06


Query file: sql\04_analytics\AN2.09_imd_missing_by_final_result.sql
Elapsed seconds: 0.017
Displayed rows: 4; truncated: False


### Interpretation and decision — AN2.09

**Observed result:** Pending local execution.

**Validation:** Record the actual returned result, any deviation from the expectation, the denominator used, and any error or timeout. Do not mark this query executed unless a result is returned.

**Decision rule:** Review the observed counts, denominators, missing values, and row grain before drawing conclusions. Retain missing final-result categories and confirm each rate's denominator; this is descriptive, not causal.

**Next step:** Retain the output and elapsed time, then decide whether a targeted follow-up check is justified.

## WO-1 acceptance criteria

WO-1 remains in progress until all eligible queries execute successfully, material deviations are investigated, interpretations and decisions are documented in `docs/analysis_log.md`, local changes are reviewed with `git diff`, and the progress ledger is updated.

Only aggregate query outputs and relevant error details are retained in the analytical record. Environment files, passwords, private connection strings, and raw CSV files are excluded.